In [ ]:
import os
import requests
import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties, fontManager

# Figure 1: Spatial coverage of Indonesia's grid points

# Repository root
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "figure_scripts":
    ROOT_DIR = ROOT_DIR.parent

# Paths
GRID_FILE = ROOT_DIR / "dataset" / "metadata" / "grid_points.csv"
OUTPUT_DIR = ROOT_DIR / "figure_scripts" / "outputs"
FONT_FILE = ROOT_DIR / "calibri.ttf"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PNG_FILE = OUTPUT_DIR / "figure2_spatial_coverage.png"
PDF_FILE = OUTPUT_DIR / "figure2_spatial_coverage.pdf"

# Configure font
if FONT_FILE.exists():
    fontManager.addfont(str(FONT_FILE))
    calibri = FontProperties(fname=str(FONT_FILE))
    plt.rcParams["font.family"] = calibri.get_name()
else:
    calibri = FontProperties(family="DejaVu Sans")
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["font.size"] = 16

# Load grid coordinates
grid = pd.read_csv(GRID_FILE)
print("Number of grid points:", len(grid))

grid_gdf = gpd.GeoDataFrame(
    grid,
    geometry=gpd.points_from_xy(
        grid["longitude"],
        grid["latitude"]
    ),
    crs="EPSG:4326"
)

# Load provincial boundaries of Indonesia
api_url = "https://www.geoboundaries.org/api/current/gbOpen/IDN/ADM1/"
response = requests.get(api_url, timeout=30)
response.raise_for_status()

boundary_url = response.json()["gjDownloadURL"]
provinces = gpd.read_file(boundary_url)

print("Number of provinces/regions:", len(provinces))

# Create spatial coverage map
fig, ax = plt.subplots(figsize=(12, 9))

provinces.plot(
    ax=ax,
    facecolor="whitesmoke",
    edgecolor="gray",
    linewidth=0.5
)

grid_gdf.plot(
    ax=ax,
    color="red",
    markersize=9,
    alpha=0.75,
    label=f"Grid points (n = {len(grid_gdf)})"
)

# Configure axes and grid
ax.set_xlabel("Longitude", fontproperties=calibri, fontsize=16)
ax.set_ylabel("Latitude", fontproperties=calibri, fontsize=16)
ax.tick_params(axis="both", labelsize=15)
ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.4)

legend_font = (
    FontProperties(fname=str(FONT_FILE), size=15)
    if FONT_FILE.exists()
    else FontProperties(family="DejaVu Sans", size=15)
)

ax.legend(
    loc="lower left",
    frameon=True,
    prop=legend_font
)

plt.tight_layout()

# Save figure in PNG and PDF formats
fig.savefig(PNG_FILE, dpi=300, bbox_inches="tight")
fig.savefig(PDF_FILE, format="pdf", bbox_inches="tight")

plt.show()
plt.close(fig)

print("Spatial coverage map saved successfully.")
print(f"PNG: {PNG_FILE}")
print(f"PDF: {PDF_FILE}")